In [14]:
import os
from PIL import Image
from torch.utils.data import Dataset

class AircraftDataset(Dataset):
    def __init__(self, images_dir, label_file, class_to_idx, transform=None):
        self.images_dir = images_dir
        self.transform = transform
        self.samples = []

        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split(" ", 1)
                img_id, family = parts[0], parts[1]
                img_path = os.path.join(images_dir, img_id + ".jpg")
                label = class_to_idx[family]
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [15]:
def build_class_index(label_file):
    families = set()
    with open(label_file, "r") as f:
        for line in f:
            _, fam = line.strip().split(" ", 1)
            families.add(fam)
    families = sorted(list(families))
    class_to_idx = {fam: i for i, fam in enumerate(families)}
    return class_to_idx, families


In [16]:
import torch
from torchvision import transforms

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [17]:
from google.colab import drive
drive.mount('/content/drive')

images_dir = "/content/drive/Shared drives/ECI271_Image_Data/data/images"
train_file = "/content/drive/Shared drives/ECI271_Image_Data/data/images_family_train.txt"
val_file = "/content/drive/Shared drives/ECI271_Image_Data/data/images_family_val.txt"
test_file = "/content/drive/Shared drives/ECI271_Image_Data/data/images_family_test.txt"

class_to_idx, families = build_class_index(train_file)

train_ds = AircraftDataset(images_dir, train_file, class_to_idx, train_tf)
val_ds   = AircraftDataset(images_dir, val_file, class_to_idx, test_tf)
test_ds  = AircraftDataset(images_dir, test_file, class_to_idx, test_tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=32)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=32)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!ls "/content/drive/Shared drives/ECI271_Image_Data/data"





families.txt			  images_manufacturer_val.txt
images				  images_test.txt
images_box.txt			  images_train.txt
images_family_test.txt		  images_val.txt
images_family_train.txt		  images_variant_test.txt
images_family_trainval.txt	  images_variant_train.txt
images_family_val.txt		  images_variant_trainval.txt
images_manufacturer_test.txt	  images_variant_val.txt
images_manufacturer_train.txt	  manufacturers.txt
images_manufacturer_trainval.txt  variants.txt


In [19]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = len(families)

# Load pretrained ViT
weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=weights)

# Replace classification head
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [20]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct = 0, 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()

        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader.dataset)
    avg_acc = total_correct / len(loader.dataset)
    return avg_loss, avg_acc


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct = 0, 0

    pbar = tqdm(loader, desc="Validating", leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            total_correct += (outputs.argmax(1) == labels).sum().item()

            pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader.dataset)
    avg_acc = total_correct / len(loader.dataset)
    return avg_loss, avg_acc

In [21]:
'mode' in globals()

False

In [22]:
epochs = 10

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")


Epoch 1/10
  Train Loss: 3.4348 | Train Acc: 0.1893
  Val Loss:   2.3343 | Val Acc:   0.4095


Epoch 2/10
  Train Loss: 1.4594 | Train Acc: 0.6434
  Val Loss:   1.3891 | Val Acc:   0.6511


Epoch 3/10
  Train Loss: 0.5338 | Train Acc: 0.9031
  Val Loss:   1.0984 | Val Acc:   0.7045


Epoch 4/10
  Train Loss: 0.1773 | Train Acc: 0.9790
  Val Loss:   0.9690 | Val Acc:   0.7381


Epoch 5/10
  Train Loss: 0.0786 | Train Acc: 0.9928
  Val Loss:   0.9200 | Val Acc:   0.7474


Epoch 6/10
  Train Loss: 0.0262 | Train Acc: 0.9997
  Val Loss:   0.7444 | Val Acc:   0.7960


Epoch 7/10
  Train Loss: 0.0095 | Train Acc: 1.0000
  Val Loss:   0.7080 | Val Acc:   0.8077


Epoch 8/10
  Train Loss: 0.0064 | Train Acc: 1.0000
  Val Loss:   0.7034 | Val Acc:   0.8137


Epoch 9/10
  Train Loss: 0.0049 | Train Acc: 1.0000
  Val Loss:   0.7067 | Val Acc:   0.8131


Epoch 10/10
  Train Loss: 0.0042 | Train Acc: 1.0000
  Val Loss:   0.6998 | Val Acc:   0.8101


In [23]:
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f"Test Accuracy: {test_acc:.4f}")


Test Accuracy: 0.8125
